# <center> TD Data Mining — DATA HUNTER CHALLENGE
##### <center> Realise par OUDIL Hiba - DATA INE2 2026/2027


**Objectif :** collecter des données issues de trois sources Web différentes, les structurer dans des DataFrames Pandas, les enregistrer en CSV, puis comparer les méthodes.

| Mission | Méthode | Source | Fichier |
|---|---|---|---|
| 1 — Static Web Scraping | Requests + BeautifulSoup | books.toscrape.com | `books.csv` |
| 2 — API Explorer | Requests + API REST | openlibrary.org | `openlibrary.csv` |
| 3 — Dynamic Web Scraping | Selenium | quotes.toscrape.com/js | `quotes.csv` |

## Installation des bibliothèques

In [1]:
!pip install requests
!pip install beautifulsoup4
!pip install pandas
!pip install selenium


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 54.1 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [selenium]7/8 [selenium]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


# Mission 1 : Static Web Scraping avec Requests et BeautifulSoup

### Import needed libraries

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

L'objectif est de récupérer le **titre**, le **prix**, la **disponibilité** et la **note** d'au moins 40 livres du site books.toscrape.com, en parcourant plusieurs pages (pagination automatique).

In [3]:
url = 'https://books.toscrape.com'

In [4]:
req = requests.get(url)

In [5]:
status = req.status_code
encoding = req.encoding
status

200

Le statut **200** signifie que la requête a réussi.

Remarque : le serveur n'indique pas l'encodage, donc `requests` choisit `ISO-8859-1` par défaut. Le symbole `£` s'afficherait alors `Â£`. On force donc l'encodage en UTF-8 :

In [6]:
print(encoding)
req.encoding = 'utf-8'
html_doc = req.text

ISO-8859-1


In [7]:
html_doc[:1000]

'<!DOCTYPE html>\n<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->\n<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->\n<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->\n<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->\n    <head>\n        <title>\n    All products | Books to Scrape - Sandbox\n</title>\n\n        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />\n        <meta name="created" content="24th Jun 2016 09:29" />\n        <meta name="description" content="" />\n        <meta name="viewport" content="width=device-width" />\n        <meta name="robots" content="NOARCHIVE,NOCACHE" />\n\n        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->\n        <!--[if lt IE 9]>\n        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>\n        <![endif]-->\n\n        \n            <link rel="shortcut icon" href

In [8]:
soup = BeautifulSoup(html_doc, 'html.parser')

In [9]:
soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

### 1. Analyse de la structure HTML

En inspectant la page (clic droit → *Inspecter*), on observe que :
- chaque livre est dans une balise `<article class="product_pod">` ;
- le **titre** complet est dans l'attribut `title` du lien `<a>` à l'intérieur de `<h3>` (le texte affiché est tronqué : « A Light in the ... ») ;
- le **prix** est dans `<p class="price_color">` (ex. `£51.77`) ;
- la **disponibilité** est dans `<p class="instock availability">` ;
- la **note** est codée dans la classe de `<p class="star-rating Three">` (One, Two, Three, Four, Five) ;
- le bouton **next** est dans `<li class="next"><a href="page-2.html">`.

Chaque page contient 20 livres, il faut donc au moins 2 pages pour 40 livres.

### Localize and collect data

In [10]:
books = soup.find_all('article', {'class': 'product_pod'})

In [11]:
len(books)

20

In [12]:
books[0]

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>

In [13]:
for item in books:
    pass

In [14]:
item

<article class="product_pod">
<div class="image_container">
<a href="catalogue/its-only-the-himalayas_981/index.html"><img alt="It's Only the Himalayas" class="thumbnail" src="media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg"/></a>
</div>
<p class="star-rating Two">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/its-only-the-himalayas_981/index.html" title="It's Only the Himalayas">It's Only the Himalayas</a></h3>
<div class="product_price">
<p class="price_color">£45.17</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>

In [15]:
# book_title : on prend l'attribut 'title' (le texte est tronqué)
title = item.find('h3').find('a').get('title')
title

"It's Only the Himalayas"

In [16]:
# book_price
price = item.find('p', {'class': 'price_color'}).text
price

'£45.17'

In [17]:
# book_availability
availability = item.find('p', {'class': 'instock availability'}).text.strip()
availability

'In stock'

In [18]:
# book_rating : la note est la 2e classe de la balise
rating = item.find('p', {'class': 'star-rating'}).get('class')
rating

['star-rating', 'Two']

In [19]:
rating = rating[1]
rating

'Two'

### Conversion des prix et des notes en valeurs numériques

- **Prix** : on garde uniquement les chiffres et le point (`£51.77` → `51.77`) puis on convertit en `float`.
- **Note** : on utilise un dictionnaire de correspondance (`'Three'` → `3`).

In [20]:
def convert_price(price_text):
    return float(re.sub(r'[^0-9.]', '', price_text))

convert_price(price)

45.17

In [21]:
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
rating_map[rating]

2

In [22]:
for item in books:
    title = item.find('h3').find('a').get('title')
    price = convert_price(item.find('p', {'class': 'price_color'}).text)
    rating = rating_map[item.find('p', {'class': 'star-rating'}).get('class')[1]]
    print(title + " --> " + str(price) + " £ --> " + str(rating) + " étoiles")

A Light in the Attic --> 51.77 £ --> 3 étoiles
Tipping the Velvet --> 53.74 £ --> 1 étoiles
Soumission --> 50.1 £ --> 1 étoiles
Sharp Objects --> 47.82 £ --> 4 étoiles
Sapiens: A Brief History of Humankind --> 54.23 £ --> 5 étoiles
The Requiem Red --> 22.65 £ --> 1 étoiles
The Dirty Little Secrets of Getting Your Dream Job --> 33.34 £ --> 4 étoiles
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull --> 17.93 £ --> 3 étoiles
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics --> 22.6 £ --> 4 étoiles
The Black Maria --> 52.15 £ --> 1 étoiles
Starving Hearts (Triangular Trade Trilogy, #1) --> 13.99 £ --> 2 étoiles
Shakespeare's Sonnets --> 20.66 £ --> 4 étoiles
Set Me Free --> 17.46 £ --> 5 étoiles
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) --> 52.29 £ --> 5 étoiles
Rip it Up and Start Again --> 35.02 £ --> 5 étoiles
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-19

### Gérer la pagination

Au lieu d'écrire les URL à la main, on suit le lien **next** tant qu'il existe et tant qu'on n'a pas atteint le nombre de livres voulu. Le lien est relatif (`page-2.html`), on reconstruit donc l'URL complète avec `urljoin`.

In [23]:
next_page = soup.find('li', {'class': 'next'})
next_page

<li class="next"><a href="catalogue/page-2.html">next</a></li>

In [24]:
next_page.find('a').get('href')

'catalogue/page-2.html'

In [25]:
from urllib.parse import urljoin
from time import sleep

url = 'https://books.toscrape.com/catalogue/page-1.html'
nb_livres_min = 40
all_books = []
page = 1

while url and len(all_books) < nb_livres_min:
    req = requests.get(url)
    if req.status_code != 200:
        print("Erreur", req.status_code, "sur", url)
        break
    req.encoding = 'utf-8'
    soup = BeautifulSoup(req.text, 'html.parser')

    for item in soup.find_all('article', {'class': 'product_pod'}):
        all_books.append({
            'title': item.find('h3').find('a').get('title'),
            'price': convert_price(item.find('p', {'class': 'price_color'}).text),
            'availability': item.find('p', {'class': 'instock availability'}).text.strip(),
            'rating': rating_map[item.find('p', {'class': 'star-rating'}).get('class')[1]]
        })
    print("Page", page, "traitée -> total :", len(all_books), "livres")

    # page suivante
    next_page = soup.find('li', {'class': 'next'})
    url = urljoin(url, next_page.find('a').get('href')) if next_page else None
    page += 1
    sleep(1)   # on reste poli avec le serveur

print("Nombre total de livres :", len(all_books))

Page 1 traitée -> total : 20 livres
Page 2 traitée -> total : 40 livres
Nombre total de livres : 40


### Organiser les résultats dans un DataFrame Pandas

In [26]:
df_books = pd.DataFrame(all_books)
df_books.head()

,title,price,availability,rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5


In [27]:
df_books.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         40 non-null     str    
 1   price         40 non-null     float64
 2   availability  40 non-null     str    
 3   rating        40 non-null     int64  
dtypes: float64(1), int64(1), str(2)
memory usage: 1.4 KB


In [28]:
df_books.describe()

,price,rating
count,40.000000,40.000000
mean,34.958750,3.050000
std,14.111908,1.501282
min,12.840000,1.000000
25%,22.575000,2.000000
50%,34.080000,3.000000
75%,50.407500,4.250000
max,57.250000,5.000000


### Save data in csv

In [29]:
df_books.to_csv('books.csv', index=False, encoding='utf-8')
pd.read_csv('books.csv').head()

,title,price,availability,rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5


# Mission 2 : API Explorer avec Requests et l'API REST Open Library

In [30]:
import requests
import json
import pandas as pd

### 1. Consultation de la documentation

D'après https://openlibrary.org/dev/docs/api/search :
- endpoint : `https://openlibrary.org/search.json`
- paramètres utiles :
  - `q` : la recherche (ici « data science ») ;
  - `fields` : les champs à renvoyer (on limite la réponse à ce qui nous intéresse) ;
  - `limit` : nombre de résultats par page ;
  - `page` : numéro de page.
- la réponse JSON contient `numFound` (nombre total de résultats) et `docs` (la liste des livres).
- champs d'un livre : `title`, `author_name` (liste), `first_publish_year`, `key` (identifiant, ex. `/works/OL...W`).

Comme dans l'exemple de la prof, on passe les paramètres avec `params=` plutôt que de les écrire dans l'URL.

In [31]:
API = "https://openlibrary.org/search.json"
params = {
    'q': 'data science',
    'fields': 'key,title,author_name,first_publish_year',
    'limit': 20,
    'page': 1
}

In [32]:
data = requests.get(API, params=params)
data.url

'https://openlibrary.org/search.json?q=data+science&fields=key%2Ctitle%2Cauthor_name%2Cfirst_publish_year&limit=20&page=1'

### Vérification du statut HTTP

In [33]:
data.status_code

200

In [34]:
if data.status_code == 200:
    data_json = data.json()
else:
    print("Erreur HTTP :", data.status_code)

In [35]:
data_json.keys()

dict_keys(['numFound', 'start', 'numFoundExact', 'num_found', 'documentation_url', 'q', 'offset', 'docs'])

In [36]:
# Nombre total de résultats disponibles
data_json['numFound']

32115

In [37]:
len(data_json['docs'])

20

In [38]:
item = data_json['docs'][0]
item

{'author_name': ['Joel Grus'],
 'first_publish_year': 2015,
 'key': '/works/OL20006690W',
 'title': 'Data science from scratch'}

In [39]:
print(json.dumps(item, indent=4))

{
    "author_name": [
        "Joel Grus"
    ],
    "first_publish_year": 2015,
    "key": "/works/OL20006690W",
    "title": "Data science from scratch"
}


In [40]:
item.keys()

dict_keys(['author_name', 'first_publish_year', 'key', 'title'])

In [41]:
item['title']

'Data science from scratch'

In [42]:
item['author_name']

['Joel Grus']

In [43]:
item['first_publish_year']

2015

In [44]:
item['key']

'/works/OL20006690W'

### Gestion des champs manquants

Certains livres n'ont pas d'auteur ou pas d'année : `item['author_name']` provoquerait alors une erreur `KeyError`. On utilise `item.get('champ', valeur_par_defaut)` qui renvoie une valeur par défaut si le champ n'existe pas. Les auteurs étant une liste, on les joint en une seule chaîne.

In [45]:
def extract_book(item):
    authors = item.get('author_name')
    return {
        'id': item.get('key', '').replace('/works/', ''),
        'title': item.get('title'),
        'authors': ', '.join(authors) if authors else None,
        'first_publish_year': item.get('first_publish_year')
    }

extract_book(item)

{'id': 'OL20006690W',
 'title': 'Data science from scratch',
 'authors': 'Joel Grus',
 'first_publish_year': 2015}

In [46]:
# Vérifions combien de livres de la 1re page ont des champs manquants
for doc in data_json['docs']:
    missing = [f for f in ['title', 'author_name', 'first_publish_year'] if f not in doc]
    if missing:
        print(doc.get('title'), '--> champs manquants :', missing)

### Gérer la pagination

Comme dans l'exemple de la World Bank, on récupère la première page puis les suivantes, jusqu'à avoir au moins 30 résultats (ou jusqu'à épuisement des résultats).

In [47]:
from time import sleep

nb_resultats_min = 30
all_docs = data_json['docs']
nb_total = data_json['numFound']

while len(all_docs) < nb_resultats_min and len(all_docs) < nb_total:
    params['page'] += 1
    response = requests.get(API, params=params)
    if response.status_code != 200:
        print("Erreur HTTP :", response.status_code)
        break
    docs = response.json()['docs']
    if not docs:        # plus de résultats
        break
    all_docs.extend(docs)
    sleep(1)

print("Nombre de résultats collectés :", len(all_docs))

Nombre de résultats collectés : 40


In [48]:
for doc in all_docs[:10]:
    print(str(doc.get('title')) + ' --> ' + str(doc.get('first_publish_year')))

Data science from scratch --> 2015
Data Science for Business --> 2013
Data Science --> 2022
Data science for dummies --> 2015
Ace the Data Science Interview --> 2021
Python Data Science Handbook --> 2016
Doing Data Science --> 2013
Business Intelligence, Analytics, and Data Science --> 2016
Big Data --> 2021
Data Science and Big Data Analytics --> 2014


### Organiser les données dans un DataFrame Pandas

In [49]:
df_ol = pd.DataFrame([extract_book(doc) for doc in all_docs])
# année en entier "nullable" (accepte les valeurs manquantes)
df_ol['first_publish_year'] = df_ol['first_publish_year'].astype('Int64')
df_ol.head()

,id,title,authors,first_publish_year
0,OL20006690W,Data science from scratch,Joel Grus,2015
1,OL17363116W,Data Science for Business,"Foster Provost, Tom Fawcett",2013
2,OL26768447W,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022
3,OL20050776W,Data science for dummies,Lillian Pierson,2015
4,OL24999191W,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021


In [50]:
df_ol.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  40 non-null     str  
 1   title               40 non-null     str  
 2   authors             40 non-null     str  
 3   first_publish_year  40 non-null     Int64
dtypes: Int64(1), str(3)
memory usage: 1.4 KB


In [51]:
# Nombre de valeurs manquantes par colonne
df_ol.isna().sum()

id                    0
title                 0
authors               0
first_publish_year    0
dtype: int64

### Save data in csv

In [52]:
df_ol.to_csv('openlibrary.csv', index=False, encoding='utf-8')
pd.read_csv('openlibrary.csv').head()

,id,title,authors,first_publish_year
0,OL20006690W,Data science from scratch,Joel Grus,2015
1,OL17363116W,Data Science for Business,"Foster Provost, Tom Fawcett",2013
2,OL26768447W,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022
3,OL20050776W,Data science for dummies,Lillian Pierson,2015
4,OL24999191W,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021


# Mission 3 : Dynamic Web Scraping avec Selenium

Sur https://quotes.toscrape.com/js/, les citations sont **générées par JavaScript** : elles ne sont pas dans le HTML renvoyé par le serveur. Vérifions-le avec Requests + BeautifulSoup :

In [53]:
import requests
from bs4 import BeautifulSoup

req = requests.get('https://quotes.toscrape.com/js/')
soup = BeautifulSoup(req.text, 'html.parser')
len(soup.find_all('div', {'class': 'quote'}))

0

Résultat : **0** citation trouvée → il faut un vrai navigateur qui exécute le JavaScript, d'où **Selenium**.

Remarque : avec Selenium 4, il n'est plus nécessaire de donner `executable_path` (l'argument utilisé dans l'exemple de la prof n'existe plus) : *Selenium Manager* télécharge automatiquement le bon driver pour Chrome.

### Import

In [54]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

### 1. Ouvrir le site avec Selenium

In [55]:
driver = webdriver.Chrome()
driver.get("https://quotes.toscrape.com/js/")

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5ad37b88008a <unknown>
#1 0x5ad37b073935 <unknown>
#2 0x5ad37b0a2d1a <unknown>
#3 0x5ad37b0a0e39 <unknown>
#4 0x5ad37b0e2613 <unknown>
#5 0x5ad37b0abc27 <unknown>
#6 0x5ad37b0aaf3e <unknown>
#7 0x5ad37b0aba11 <unknown>
#8 0x5ad37b841a3c <unknown>
#9 0x5ad37b84027b <unknown>
#10 0x5ad37b82af25 <unknown>
#11 0x5ad37b840eaa <unknown>
#12 0x5ad37b8133e9 <unknown>
#13 0x5ad37b869758 <unknown>
#14 0x5ad37b8698f5 <unknown>
#15 0x5ad37b87e922 <unknown>
#16 0x7d29b7727b84 <unknown>
#17 0x7d29b77b4ecc <unknown>


### 2. Attendre le chargement des citations

Au lieu d'un `sleep(10)` fixe comme dans l'exemple eBay, on utilise **WebDriverWait** : il attend (au maximum 10 s) que les éléments `div.quote` soient présents, et continue dès qu'ils le sont.

In [ ]:
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))

### Localise data

En inspectant la page :
- chaque citation est dans `<div class="quote">` ;
- le texte dans `<span class="text">` ;
- l'auteur dans `<small class="author">` ;
- les tags dans des liens `<a class="tag">` à l'intérieur de `<div class="tags">` ;
- le bouton suivant dans `<li class="next"><a>`.

In [ ]:
results = driver.find_elements(By.CLASS_NAME, 'quote')

In [ ]:
len(results)

In [ ]:
for quote in results:
    pass

In [ ]:
quote.text

In [ ]:
# Get the text of the quote
text = quote.find_element(By.CLASS_NAME, "text").text
print("Citation : ", text)

In [ ]:
# Get the author
author = quote.find_element(By.CLASS_NAME, "author").text
print("Auteur : ", author)

In [ ]:
# Get the tags (plusieurs éléments -> find_elements)
tags = [t.text for t in quote.find_elements(By.CLASS_NAME, "tag")]
print("Tags : ", tags)

### Interacting with the page : Clicking Next Page Using Selenium

Pour les deux premières pages, on :
1. attend les citations avec WebDriverWait ;
2. extrait les données ;
3. clique sur **Next** et attend que l'ancienne page disparaisse (`staleness_of`) pour être sûr de ne pas relire la même page.

Le bloc `try / finally` garantit que le navigateur est **fermé correctement** avec `driver.quit()`, même si une erreur survient.

In [ ]:
driver = webdriver.Chrome()
all_quotes = []
nb_pages = 2

try:
    driver.get("https://quotes.toscrape.com/js/")
    wait = WebDriverWait(driver, 10)

    for page in range(1, nb_pages + 1):
        # attendre le chargement des citations
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
        results = driver.find_elements(By.CLASS_NAME, "quote")

        for quote in results:
            all_quotes.append({
                'quote': quote.find_element(By.CLASS_NAME, "text").text,
                'author': quote.find_element(By.CLASS_NAME, "author").text,
                'tags': ', '.join(t.text for t in quote.find_elements(By.CLASS_NAME, "tag")),
                'page': page
            })
        print("Page", page, ":", len(results), "citations")

        # aller à la page suivante (sauf après la dernière page voulue)
        if page < nb_pages:
            next_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
            first_quote = results[0]
            next_page.click()
            wait.until(EC.staleness_of(first_quote))
finally:
    driver.quit()

print("Nombre total de citations :", len(all_quotes))

### Organiser les données dans un DataFrame et Save data in csv

In [ ]:
df_quotes = pd.DataFrame(all_quotes)
df_quotes.head()

In [ ]:
df_quotes.info()

In [ ]:
df_quotes.to_csv('quotes.csv', index=False, encoding='utf-8')
pd.read_csv('quotes.csv').head()

# Comparaison des trois méthodes

| Critère | Requests + BeautifulSoup | API REST | Selenium |
|---|---|---|---|
| Type de contenu | HTML statique | JSON structuré | HTML généré par JavaScript |
| Vitesse | Rapide | Très rapide | Lente (lance un navigateur) |
| Difficulté | Moyenne (analyser le HTML) | Faible (lire la documentation) | Plus élevée (driver, attentes) |
| Robustesse | Fragile si le HTML change | Stable (format documenté) | Fragile si le HTML change |
| Données | À nettoyer (`£51.77`, `"Three"`) | Déjà typées, mais champs manquants | À nettoyer (guillemets, tags) |
| Pagination | Suivre le lien *next* | Paramètre `page` | Cliquer sur *Next* |
| Quand l'utiliser | Pages statiques sans API | Dès qu'une API existe | Contenu dynamique / interactions |

**Conclusion :** l'API est la méthode à privilégier quand elle existe ; BeautifulSoup convient aux pages statiques ; Selenium n'est utile que lorsque le contenu dépend du JavaScript ou d'interactions (clics, formulaires).

# Question de conclusion

### Les données extraites sont-elles directement exploitables pour le Data Mining ?

**Non, pas entièrement.** Elles sont structurées en tableaux, mais elles présentent encore des défauts :
- `books.csv` : la disponibilité est un texte (« In stock ») et non une variable binaire ; on n'a pas la quantité en stock ;
- `openlibrary.csv` : des valeurs manquantes (auteurs, années), des doublons possibles (même livre en plusieurs éditions), plusieurs auteurs dans une même cellule ;
- `quotes.csv` : les citations contiennent des guillemets typographiques (“ ”), les tags sont regroupés dans une seule chaîne de caractères.

### Trois opérations de prétraitement

1. **Gestion des valeurs manquantes et des doublons** : supprimer les doublons (`drop_duplicates`), puis supprimer ou imputer les valeurs manquantes (ex. année de publication manquante dans Open Library).
2. **Nettoyage et transformation des types** : convertir la disponibilité en variable binaire (1/0), enlever les guillemets et espaces des citations, normaliser le texte (minuscules, accents), vérifier que prix, notes et années sont bien numériques.
3. **Encodage et normalisation** : transformer les variables catégorielles ou multi-valeurs en variables numériques (ex. *one-hot encoding* des tags ou des auteurs), et normaliser les variables numériques (ex. prix avec une normalisation Min-Max) pour que les algorithmes ne soient pas biaisés par les échelles.

In [ ]:
# Exemple d'application de ces prétraitements
df_books = pd.read_csv('books.csv')
df_books['in_stock'] = df_books['availability'].str.contains('In stock').astype(int)
df_books['price_norm'] = (df_books['price'] - df_books['price'].min()) / (df_books['price'].max() - df_books['price'].min())

df_ol = pd.read_csv('openlibrary.csv').drop_duplicates(subset='id')
print("Valeurs manquantes Open Library :\n", df_ol.isna().sum())

df_quotes = pd.read_csv('quotes.csv')
df_quotes['quote'] = df_quotes['quote'].str.strip('“”')
tags_dummies = df_quotes['tags'].fillna('').str.get_dummies(sep=', ')

df_books.head()